In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
for candidate in (repo_root, repo_root / "src"):
    candidate_str = str(candidate)
    if candidate.exists() and candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)


In [ ]:
from pathlib import Path
import csv

from src.drive_service.logging_utils import setup_logging
from src.pipeline_paths import build_pipelines_paths
from src.filter_midnight_events_from_days_raw import filter_midnight_events_dir


In [ ]:
root = "1FUosjKncLt18JzojmX8tKQm1nbgPI133"
paths = build_pipelines_paths(root)

events_name = "*.events_from_days_raw.csv"
events_files = sorted(Path(paths.events_output).rglob(events_name))
if not events_files:
    raise FileNotFoundError(
        f"No events files found in {paths.events_output} with pattern {events_name}"
    )

paths.events_output, len(events_files), events_files[:5]


In [ ]:
verbose = True
out_name = "events_from_days_raw.cleaned.csv"
report_json = paths.events_output / "events_from_days_raw.clean_midnight.report.json"
removed_csv_path = paths.events_output / "events_from_days_raw.midnight_removed.csv"

max_removed_examples_per_file = 10

setup_logging(verbose)

report = filter_midnight_events_dir(
    input_dir=str(paths.events_output),
    events_name=events_name,
    out_name=out_name,
    report_json=str(report_json),
    removed_csv=str(removed_csv_path),
    max_removed_examples_per_file=max_removed_examples_per_file,
)

report["stats"]


In [ ]:
cleaned_pattern = f"*.{out_name}"
cleaned_files = sorted(Path(paths.events_output).rglob(cleaned_pattern))
len(cleaned_files), cleaned_files[:5]


In [ ]:
if cleaned_files:
    sample_cleaned = cleaned_files[0]
    with open(sample_cleaned, "r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        sample_rows = []
        for i, row in enumerate(reader):
            sample_rows.append(row)
            if i >= 10:
                break
    sample_cleaned, sample_rows
else:
    "No cleaned events CSV files generated"


In [ ]:
files_with_removed = report.get("files_with_removed", [])
file_errors = report.get("file_errors", [])

removed_rows_preview = []
if removed_csv_path.exists():
    with open(removed_csv_path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        for i, row in enumerate(reader):
            removed_rows_preview.append(row)
            if i >= 10:
                break

{
    "report_json": str(report_json),
    "removed_csv": str(removed_csv_path),
    "files_with_removed_count": len(files_with_removed),
    "file_errors_count": len(file_errors),
    "files_with_removed_preview": files_with_removed[:3],
    "file_errors_preview": file_errors[:3],
    "removed_rows_preview": removed_rows_preview,
}
